# Notebook 09: Megatron-LM Patterns

## Why This Matters

Megatron-LM is the training framework behind GPT-3, many of the LLaMA variants, Falcon, and most frontier models. It defines the canonical patterns for combining tensor parallelism, pipeline parallelism, and data parallelism into what is called "3D parallelism." Engineers working on LLM infrastructure are expected to understand Megatron's sequence parallel attention, how its BERT/GPT model definitions differ from standard PyTorch implementations, and why it achieves near-linear scaling to thousands of GPUs.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. 3D Parallelism: TP × PP × DP

Megatron-LM combines three orthogonal parallelism strategies:

**Tensor Parallelism (TP)**: within a node, across NVLink-connected GPUs. Splits individual layers (attention heads, FFN columns/rows). Degree: typically 4 or 8.

**Pipeline Parallelism (PP)**: across nodes. Each node handles a contiguous set of transformer layers. Communication: send activation tensors between nodes via InfiniBand. Degree: 2-16 nodes.

**Data Parallelism (DP)**: replicate the full model (post-TP+PP) across independent data parallel replicas. Degree: number of replicas.

Total GPUs: `TP × PP × DP`

Example for GPT-3 (175B parameters) on 1024 A100s:
- TP=8 (within node)
- PP=16 (across 16 node groups)
- DP=8 (8 replicas)
- Total: 8 × 16 × 8 = 1024 GPUs

In [ ]:
# 3D parallelism grid visualization

def visualize_3d_parallelism(tp, pp, dp):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    total_gpus = tp * pp * dp
    
    colors = plt.cm.Set1(np.linspace(0, 0.8, dp))
    
    for dp_rank in range(dp):
        for pp_rank in range(pp):
            for tp_rank in range(tp):
                x, y, z = tp_rank, pp_rank, dp_rank
                ax.scatter(x, y, z, 
                          c=[colors[dp_rank]], 
                          s=200, 
                          edgecolors='black',
                          linewidths=0.5)
                
                # TP connections (within node, x-axis)
                if tp_rank < tp - 1:
                    ax.plot([x, x+1], [y, y], [z, z], 'b-', alpha=0.4, linewidth=2)
                
                # PP connections (across nodes, y-axis)
                if pp_rank < pp - 1:
                    ax.plot([x, x], [y, y+1], [z, z], 'r-', alpha=0.3, linewidth=1)
    
    ax.set_xlabel('Tensor Parallel Rank', fontsize=10)
    ax.set_ylabel('Pipeline Stage', fontsize=10)
    ax.set_zlabel('Data Parallel Rank', fontsize=10)
    ax.set_title(f'3D Parallelism: TP={tp}, PP={pp}, DP={dp}\n'
                 f'Total GPUs: {total_gpus}', fontsize=12)
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='b', linewidth=2, label=f'TP (NVLink, within node)'),
        Line2D([0], [0], color='r', linewidth=1, label=f'PP (InfiniBand, across nodes)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
               markersize=10, label=f'DP replica (gradient all-reduce)'),
    ]
    ax.legend(handles=legend_elements, loc='upper left')
    
    plt.tight_layout()
    plt.savefig('3d_parallelism.png', dpi=100)
    plt.show()

visualize_3d_parallelism(tp=4, pp=4, dp=2)
print(f"Total GPUs: 4 × 4 × 2 = {4*4*2}")
print("\nEach 'column' (same x,z, different y) is one pipeline.")
print("Each 'row' (same y,z, different x) is one tensor-parallel group.")
print("Each 'depth' (same x,y, different z) is one data-parallel replica.")

## 2. Megatron's Sequence Parallelism

Standard tensor parallelism leaves LayerNorm and Dropout operating on the full tensor -- these operations are memory-intensive and not parallelized. Megatron v3 introduces **sequence parallelism** to fix this.

In sequence parallel mode:
- **Non-tensor-parallel operations** (LayerNorm, Dropout) operate on a sequence shard: each GPU handles `seq_len / TP` tokens
- Before the TP attention/FFN, an **all-gather** reconstructs the full sequence on each GPU
- After the TP computation, a **reduce-scatter** shards the result back across the sequence dimension

This eliminates the memory spike from non-TP operations and reduces peak activation memory by another TP factor.

In [ ]:
# Sequence parallelism: memory analysis

def activation_memory_mb(batch, seq_len, d_model, n_layers, dtype_bytes=2, tp=1):
    """
    Estimate peak activation memory with and without sequence parallelism.
    Non-TP ops (LayerNorm): batch * seq * d_model * n_layers
    TP attention/FFN: batch * (seq/tp) * d_model * n_layers (each GPU)
    """
    # Without sequence parallelism: LayerNorm uses full seq
    layernorm_mem = batch * seq_len * d_model * n_layers * dtype_bytes
    attn_ffn_mem = batch * (seq_len // tp) * d_model * n_layers * dtype_bytes
    total_without_sp = (layernorm_mem + attn_ffn_mem) / 1e6  # MB
    
    # With sequence parallelism: all ops use seq/tp
    total_with_sp = batch * (seq_len // tp) * d_model * n_layers * dtype_bytes / 1e6
    
    return total_without_sp, total_with_sp

batch, seq_len, d_model, n_layers = 4, 4096, 8192, 96

print(f"Model: d_model={d_model}, seq_len={seq_len}, batch={batch}, n_layers={n_layers}")
print()
print(f"{'TP Degree':<12} {'Without SP (GB)':<20} {'With SP (GB)':<20} {'Savings'}")
print("-" * 60)

for tp in [1, 2, 4, 8]:
    mem_no_sp, mem_sp = activation_memory_mb(batch, seq_len, d_model, n_layers, tp=tp)
    savings = (mem_no_sp - mem_sp) / mem_no_sp
    print(f"{tp:<12} {mem_no_sp/1e3:<20.1f} {mem_sp/1e3:<20.1f} {savings:.0%}")

print()
print("With SP, LayerNorm/Dropout see only seq/TP tokens -- saves TP factor in peak memory.")
print("Communication cost: 2 extra all-gathers (for SP<->TP transitions) per layer.")

## 3. Megatron's Model Architecture Choices

Megatron makes specific architectural choices that differ from standard PyTorch:

**Pre-norm vs post-norm**: Megatron uses pre-LayerNorm (LayerNorm before attention/FFN), which is more training-stable at large scales. Most modern LLMs follow this.

**Fused kernels**: custom CUDA kernels for:
- `fused_softmax`: softmax + attention masking in one kernel pass
- `fused_bias_gelu`: bias add + GeLU activation fused
- Apex FusedLayerNorm: faster than PyTorch's default LayerNorm

**Gradient clipping**: Megatron clips global norm across the full model, not per-layer. This requires all-reducing the squared gradient norms across all TP/PP ranks before clipping.

**Vocabulary parallelism**: for large vocabularies, the embedding and output projection are also column-parallel (split vocabulary across TP ranks).

In [ ]:
# Megatron-style transformer layer (simplified, CPU-compatible)
import torch.nn.functional as F

class MegatronTransformerLayer(nn.Module):
    """
    Simplified Megatron-style transformer layer with:
    - Pre-layer norm
    - Column-parallel attention (simulated with regular MHA here)
    - Column+row parallel FFN
    - Fused bias+GeLU
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        # Pre-norm (Megatron uses this, not post-norm)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Attention projections (would be column-parallel in real Megatron)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        
        # FFN (would be column+row parallel in real Megatron)
        self.ffn_fc1 = nn.Linear(d_model, d_ff, bias=True)
        self.ffn_fc2 = nn.Linear(d_ff, d_model, bias=True)
        
        self.dropout = nn.Dropout(dropout)
    
    def _attention(self, x):
        B, S, D = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        q = q.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        
        scale = self.d_head ** -0.5
        attn = torch.softmax(q @ k.transpose(-2, -1) * scale, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(out)
    
    def _ffn(self, x):
        # Fused bias+GeLU (in real Megatron, this is a custom CUDA kernel)
        h = F.gelu(self.ffn_fc1(x))
        return self.ffn_fc2(h)
    
    def forward(self, x):
        # Pre-norm attention
        x = x + self.dropout(self._attention(self.norm1(x)))
        # Pre-norm FFN
        x = x + self.dropout(self._ffn(self.norm2(x)))
        return x

# Test
torch.manual_seed(42)
model = MegatronTransformerLayer(d_model=256, n_heads=8, d_ff=1024)
x = torch.randn(2, 16, 256)
out = model(x)
print(f"Input: {x.shape} -> Output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print()
print("Key differences from stock PyTorch TransformerEncoderLayer:")
print("  1. Pre-LayerNorm (more stable at large scale)")
print("  2. Separate Q/K/V projection (enables head parallelism)")
print("  3. Bias in QKV/output projections (Megatron default)")
print("  4. In practice: fused CUDA kernels for each sub-operation")

## 4. Scaling Laws and Optimal Model Size

Chinchilla scaling laws (Hoffmann et al. 2022) tell us the optimal model size and training tokens for a given compute budget:

$$N_{optimal} \approx \frac{C}{6D}$$

where:
- N = number of parameters
- D = number of training tokens
- C = total FLOP budget (6ND FLOPs per forward+backward)

**Key insight**: the original GPT-3 (175B params, 300B tokens) was significantly undertrained. Chinchilla predicts that with GPT-3's compute budget, a ~70B model trained on ~1.4T tokens would be better. This is exactly what Chinchilla (70B, 1.4T tokens) showed.

Understanding scaling laws is critical for deciding where to invest compute: train a bigger model for fewer tokens, or a smaller model for more tokens?

In [ ]:
# Chinchilla scaling law analysis

def compute_budget_flops(n_params, n_tokens):
    """FLOPs for training (approx): 6 * N * D"""
    return 6 * n_params * n_tokens

def chinchilla_optimal(compute_budget_flops):
    """
    Optimal N and D given compute budget C.
    Chinchilla: N = D (roughly), C = 6*N*D
    So N_opt = D_opt = sqrt(C/6)
    """
    n_opt = (compute_budget_flops / 6) ** 0.5
    d_opt = n_opt
    return n_opt, d_opt

# Notable models
models = {
    'GPT-3': (175e9, 300e9),
    'Chinchilla': (70e9, 1.4e12),
    'LLaMA-1 7B': (7e9, 1e12),
    'LLaMA-2 7B': (7e9, 2e12),
    'LLaMA-2 70B': (70e9, 2e12),
    'Mistral 7B': (7e9, 1e12),
}

print(f"{'Model':<20} {'Params':<12} {'Tokens':<12} {'Compute (ZFLOPs)':<20} {'Optimal N':<15} {'Token ratio'}")
print("-" * 90)

for name, (n, d) in models.items():
    c = compute_budget_flops(n, d)
    n_opt, d_opt = chinchilla_optimal(c)
    ratio = d / n  # tokens per parameter
    zflops = c / 1e21
    print(f"{name:<20} {n/1e9:<12.0f} {d/1e12:<12.1f} {zflops:<20.2f} {n_opt/1e9:<15.0f} {ratio:<.0f}")

print()
# Plot optimal frontier
compute_budgets = np.logspace(20, 26, 100)
n_opts = [(c/6)**0.5 / 1e9 for c in compute_budgets]

plt.figure(figsize=(10, 6))
plt.loglog(compute_budgets, n_opts, 'b-', linewidth=2, label='Chinchilla optimal N')

for name, (n, d) in models.items():
    c = compute_budget_flops(n, d)
    plt.scatter(c, n/1e9, s=100, zorder=5)
    plt.annotate(name, xy=(c, n/1e9), xytext=(c*1.5, n/1e9*1.2), fontsize=8)

plt.xlabel('Compute budget (FLOPs)')
plt.ylabel('Optimal model size (B parameters)')
plt.title('Chinchilla Scaling Laws: Optimal Model Size vs Compute')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('scaling_laws.png', dpi=100)
plt.show()

## Summary

### Key Concepts

| Pattern | What it does | Memory saving | Communication cost |
|---------|-------------|---------------|-------------------|
| TP=8 | Split attention heads + FFN columns | 8x per layer | 2 all-reduces/layer |
| PP=16 | 16 pipeline stages across nodes | 16x in total params | activation tensors between stages |
| SP (sequence parallel) | Shard LayerNorm/Dropout by seq dim | TP factor in non-TP ops | 2 extra all-gathers/layer |
| Vocab parallel | Split embedding table | vocab_size / TP | 1 all-reduce after embedding |

**Megatron's key recipe**:
1. Fuse column-parallel + row-parallel FFN into 1 all-reduce
2. Use sequence parallelism to shard non-TP ops
3. Pre-norm for training stability
4. Pack TP within NVLink node, PP across InfiniBand nodes
5. DP across independent replicas for throughput

**Next**: `10_gradient_checkpointing.ipynb` -- how to train larger models by recomputing activations instead of storing them.